# GlassCut Slide Library - Usage Guide

Welcome to GlassCut! This notebook demonstrates how to work with histopathology slides using the GlassCut library.

**What you'll learn:**
- Loading and exploring whole slide images (WSIs)
- Downloading samples from TCGA dataset
- Applying different tiling strategies
- Batch processing multiple slides
- Stain normalization and tissue detection

Let's get started!

In [1]:
# Import main libraries
import plotly.graph_objects as go
from glasscut.slides import Slide
from glasscut.data import breast_tissue

/home/camilosinning/Documents/GlassCut/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download a breast tissue slide (or any tissue type)
# The download is cached, so it only happens once
slide_obj, slide_path = breast_tissue()

print(f"\n✓ Slide ready at: {slide_path}")
print(f"  Size: {slide_path.stat().st_size / (1024**2):.1f} MB")


✓ Slide ready at: /home/camilosinning/.cache/glasscut-data/tcga/breast/TCGA-A8-A082-01A-01-TS1.3cad4a77-47a6-4658-becf-d8cffa161d3a.svs
  Size: 285.2 MB


In [3]:
# Load the slide
slide = Slide(str(slide_path))

# Explore slide properties
print(f"Slide Name: {slide.name}")
print(f"Slide Shape: {slide.dimensions} pixels")
print(f"Available Magnifications: {slide.magnifications}")
print(f"\nSlide Resolution (µm/pixel): {slide.mpp:.4f}")

Slide Name: TCGA-A8-A082-01A-01-TS1.3cad4a77-47a6-4658-becf-d8cffa161d3a
Slide Shape: (96972, 30681) pixels
Available Magnifications: [40.0, 20.0, 10.0, 5.0]

Slide Resolution (µm/pixel): 0.2505


In [6]:
# Get a thumbnail view
fig = go.Figure(data=go.Image(z=slide.thumbnail))
fig.update_layout(
    title=f"Slide Thumbnail - {slide.name}",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    height=600,
    width=800
)
fig.show()

print(f"Thumbnail size: {slide.thumbnail.size}")

Thumbnail size: (968, 306)


<h1 style="color:red">I'm here</h1>

In [5]:
# Get a region of interest at different magnifications
print("Reading regions at different zoom levels...")

# Read a 1024x1024 pixel region at different magnifications
x, y, w, h = 2000, 2000, 1024, 1024

regions = {}
for mag in [10, 20, 40]:
    level = slide.get_level_for_magnification(mag)
    region = slide.read_region((x, y), mag, (w, h))
    regions[mag] = region
    print(f"  {mag}x magnification (Level {level}): {region.size}")

# Visualize regions with Plotly subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{mag}x Magnification" for mag in [10, 20, 40]]
)

for i, (mag, region) in enumerate(regions.items(), 1):
    fig.add_trace(
        go.Image(z=region),
        row=1, col=i
    )

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=400, showlegend=False)
fig.show()

Reading regions at different zoom levels...


AttributeError: 'Slide' object has no attribute 'get_level_for_magnification'

## 5. Tissue Detection

Before tiling, we can detect tissue regions to avoid extracting tiles from background areas.

In [ ]:
# Get tissue mask using Otsu's method
tissue_mask = slide.get_tissue_mask()

print(f"Tissue mask shape: {tissue_mask.shape}")
print(f"Tissue coverage: {(tissue_mask.sum() / tissue_mask.size * 100):.1f}%")

# Visualize the mask with Plotly
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Original Slide", "Tissue Mask"]
)

fig.add_trace(go.Image(z=slide.thumbnail), row=1, col=1)
fig.add_trace(go.Image(z=tissue_mask, colorscale="Gray"), row=1, col=2)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 6. Tiling Strategies

GlassCut provides multiple tiling strategies. Let's explore them:

### 6.1 Grid Tiling
Extract tiles in a regular grid pattern across the tissue areas.

In [ ]:
from glasscut.tiler import GridTiler

# Create a grid tiler configuration
grid_tiler = GridTiler(
    tile_size=(512, 512),        # 512x512 pixel tiles
    overlap=0.1,                 # 10% overlap between tiles
    min_tissue_ratio=0.8,        # At least 80% tissue in tile
    save_empty=False             # Don't save tiles with little/no tissue
)

print(f"Grid Tiler Configuration:")
print(f"  Tile size: {grid_tiler.tile_size}")
print(f"  Overlap: {grid_tiler.overlap * 100:.0f}%")
print(f"  Min tissue ratio: {grid_tiler.min_tissue_ratio * 100:.0f}%")

# Extract tiles (we'll just demonstrate with coordinates, not save)
tiles = grid_tiler.extract(slide, save_dir=None)
print(f"\nWould extract ~{len(tiles)} tiles in grid pattern")
print(f"First few tile positions: {tiles[:3]}")

### 6.2 Random Tiling
Extract a fixed number of random tiles from tissue regions.

In [ ]:
from glasscut.tiler import RandomTiler

# Create a random tiler configuration
random_tiler = RandomTiler(
    num_tiles=50,                # Extract 50 random tiles
    tile_size=(256, 256),        # 256x256 pixel tiles
    seed=42,                     # For reproducibility
    min_tissue_ratio=0.7,        # At least 70% tissue in tile
    max_attempts=1000            # Maximum attempts to find valid tiles
)

print(f"Random Tiler Configuration:")
print(f"  Target tiles: {random_tiler.num_tiles}")
print(f"  Tile size: {random_tiler.tile_size}")
print(f"  Seed: {random_tiler.seed}")
print(f"  Min tissue ratio: {random_tiler.min_tissue_ratio * 100:.0f}%")

# Extract random tiles
tiles_random = random_tiler.extract(slide, save_dir=None)
print(f"\nExtracted {len(tiles_random)} random tiles")
print(f"First few tile positions: {tiles_random[:3]}")

### 6.3 Conditional Tiling (Tissue-based)
Extract tiles adaptively based on tissue quality and characteristics.

In [ ]:
from glasscut.tiler import ConditionalTiler
from glasscut.tissue_detectors import OtsuTissueDetector

# Create a tissue detector
detector = OtsuTissueDetector()

# Create a conditional tiler
conditional_tiler = ConditionalTiler(
    tissue_detector=detector,
    tile_size=(512, 512),
    overlap=0.0,                # No overlap for conditional
    min_tissue_in_tile=0.75,    # 75% tissue required
    mask_level=5                # Use lower resolution for mask
)

print(f"Conditional Tiler Configuration:")
print(f"  Tile size: {conditional_tiler.tile_size}")
print(f"  Min tissue ratio: {conditional_tiler.min_tissue_in_tile * 100:.0f}%")
print(f"  Mask level: {conditional_tiler.mask_level}")

# Extract tiles based on tissue characteristics
tiles_conditional = conditional_tiler.extract(slide, save_dir=None)
print(f"\nExtracted {len(tiles_conditional)} tissue-aware tiles")

## 7. Stain Normalization

Normalize staining variations across slides for consistent downstream analysis.

In [ ]:
from glasscut.stain_normalizers import MacenkoNormalizer
from plotly.subplots import make_subplots

# Create a sample region for demonstration
sample_region = slide.read_region((5000, 5000), 20, (512, 512))

# Create normalizer
normalizer = MacenkoNormalizer()

# Normalize the region
normalized = normalizer.normalize(np.array(sample_region))

# Visualize with Plotly
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Original Region", "Macenko Normalized"]
)

fig.add_trace(go.Image(z=sample_region), row=1, col=1)
fig.add_trace(go.Image(z=normalized.astype(np.uint8)), row=1, col=2)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=400, showlegend=False)
fig.show()

## 8. Batch Processing Multiple Slides

Now let's explore how to process multiple slides in a batch—the foundation for dataset creation.

### 8.1 Download Multiple Slides

In [ ]:
# Download multiple tissue types from the available samples
print("Downloading sample slides...\n")

# Available tissue loaders from glasscut.data
from glasscut.data import (
    aorta_tissue,
    heart_tissue,
    cmu_small_region,
    ihc_breast,
    ihc_kidney,
)

# Download one sample of each tissue type
tissue_loaders = {
    "aorta": aorta_tissue,
    "heart": heart_tissue,
    "cmu": cmu_small_region,
    "ihc_breast": ihc_breast,
    "ihc_kidney": ihc_kidney,
}

downloaded_slides = {}

for tissue_name, loader in tissue_loaders.items():
    try:
        slide_obj, slide_path = loader()
        downloaded_slides[tissue_name] = slide_path
        print(f"  ✓ {tissue_name}: {slide_path.name}")
    except Exception as e:
        print(f"  ✗ {tissue_name}: {e}")

print(f"\nTotal downloaded: {len(downloaded_slides)} slides")

### 8.2 Process All Downloaded Slides

Now let's extract tiles from all downloaded slides and collect statistics.

In [ ]:
# Process all downloaded slides
print("Processing downloaded slides...\n")

from glasscut.tiler import RandomTiler

processing_results = {}

for tissue, slide_path in downloaded_slides.items():
    print(f"Processing {tissue.upper()} sample: {slide_path.name}")
    
    try:
        # Load slide
        slide_obj = Slide(str(slide_path))
        
        # Extract random tiles for dataset
        tiler = RandomTiler(
            num_tiles=5,           # Just 5 tiles per slide for demo
            tile_size=(256, 256),
            seed=42
        )
        
        tiles = tiler.extract(slide_obj, save_dir=None)
        
        # Collect stats
        processing_results[tissue] = {
            "slide_path": slide_path,
            "dimensions": slide_obj.dimensions,
            "num_tiles": len(tiles),
            "magnifications": slide_obj.magnifications,
        }
        
        print(f"  ✓ Slide: {slide_obj.dimensions}")
        print(f"  ✓ Extracted {len(tiles)} tiles")
        print()
        
    except Exception as e:
        print(f"  ✗ Error: {e}\n")

# Display summary
print("\n" + "="*60)
print("BATCH PROCESSING SUMMARY")
print("="*60)

for tissue, results in processing_results.items():
    print(f"\n{tissue.upper()}:")
    print(f"  Dimensions: {results['dimensions']}")
    print(f"  Tiles extracted: {results['num_tiles']}")
    print(f"  Available magnifications: {results['magnifications']}")

## 9. Advanced: Full Dataset Processing with DatasetGenerator

For production workflows, GlassCut provides a `DatasetGenerator` for automated, parallel batch processing of entire datasets.

This is useful when you have:
- Multiple slides (100s or 1000s)
- Need parallel processing for speed
- Want standardized output organization
- Need metadata tracking

In [ ]:
# Example of how to use DatasetGenerator for production workflows
# (We'll show the code structure, not execute full download)

code_example = """
from glasscut.dataset import DatasetGenerator, DatasetConfig
from glasscut.tiler import RandomTiler
from pathlib import Path

# Configure the dataset generation
config = DatasetConfig(
    dataset_id="tcga_breast_pilot",
    output_dir="./datasets/tcga_breast",
    tiler=RandomTiler(
        num_tiles=100,
        tile_size=(512, 512),
        seed=42
    ),
    num_workers=4,  # Parallel processing with 4 workers
    tile_format="png",
    save_metadata=True
)

# Create generator
generator = DatasetGenerator(config)

# Process all slides
slide_paths = [
    Path("slide_001.svs"),
    Path("slide_002.svs"),
    Path("slide_003.svs"),
    # ... more slides
]

# Process dataset (returns dataset info)
dataset_info = generator.process_dataset(slide_paths)

print(f"Generated dataset: {dataset_info['dataset_id']}")
print(f"Total tiles: {dataset_info['total_tiles']}")
print(f"Output location: {dataset_info['output_dir']}")
"""

print("Example: Processing entire dataset")
print("=" * 60)
print(code_example)

## 10. Next Steps

You now understand the core GlassCut workflow:

1. **Explore data**: Download samples from TCGA
2. **Analyze slides**: Load and examine with Slide class
3. **Extract tiles**: Use tiling strategies (Grid, Random, Conditional)
4. **Normalize**: Apply stain normalization if needed
5. **Scale up**: Use DatasetGenerator for batch processing

### For your project:
- Modify tiling parameters based on your downstream task
- Experiment with tissue detection thresholds
- Use batch processing for large-scale dataset generation
- Organize results with consistent metadata

### Learn more:
- GlassCut documentation: `docs/`
- Tiler guide: `docs/TILER_GUIDE.md`
- Dataset generation: `docs/DATASET_GENERATION.md`
- Examples: `examples/`

In [ ]:
print("""
╔════════════════════════════════════════════════════════════╗
║         Successfully completed GlassCut Tutorial!          ║
║                                                            ║
║  You've learned how to:                                  ║
║  • Access TCGA datasets                                  ║
║  • Load and explore WSIs                                 ║
║  • Apply multiple tiling strategies                      ║
║  • Normalize staining variations                         ║
║  • Process slides in batch                               ║
║                                                            ║
║  Ready to build your dataset! 🔬                         ║
╚════════════════════════════════════════════════════════════╝
""")